# 00 — Colab environment check
No dissertation model is trained in this notebook. TEST is not accessed.

In [ ]:
REQUIRE_GPU = True
REPO_URL = "REPLACE_WITH_GITHUB_REPO_URL"
DRIVE_ROOT = "/content/drive/MyDrive/evo2_dissertation"
REPO_ROOT = "/content/evo2-dissertation"
assert REPO_URL.startswith(('https://', 'git@')) and 'REPLACE_' not in REPO_URL, 'Set REPO_URL first'


In [ ]:
import os, platform, shutil, subprocess, time, torch
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print('VRAM GiB:', torch.cuda.get_device_properties(0).total_memory/2**30 if torch.cuda.is_available() else 0)
print('Disk free GiB:', shutil.disk_usage('/content').free/2**30)
if REQUIRE_GPU and not torch.cuda.is_available(): raise RuntimeError('Enable a Colab GPU runtime')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
assert os.path.isdir(DRIVE_ROOT), f'Missing Drive payload: {DRIVE_ROOT}'


In [ ]:
if os.path.isdir(os.path.join(REPO_ROOT, '.git')):
    subprocess.run(['git','-C',REPO_ROOT,'pull','--ff-only'], check=True)
else:
    subprocess.run(['git','clone',REPO_URL,REPO_ROOT], check=True)
subprocess.run(['pip','install','-q','-r',f'{REPO_ROOT}/requirements/requirements-colab.txt'], check=True)
subprocess.run(['pip','install','-q','-e',REPO_ROOT], check=True)
subprocess.run(['python',f'{REPO_ROOT}/scripts/verify_environment.py','--require-gpu'], check=True)


In [ ]:
size = 4096
a = torch.randn(size, size, device='cuda'); b = torch.randn(size, size, device='cuda')
torch.cuda.synchronize(); started=time.perf_counter(); c=a@b; torch.cuda.synchronize()
print('Simple GPU matrix-op seconds:', time.perf_counter()-started)
print('Drive mount: PASS; Git clone: PASS; package install: PASS; dissertation training: NOT RUN')
